In [2]:
import numpy as np
import pandas as pd
import joblib
import json
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'plotly_mimetype'
import warnings
warnings.filterwarnings('ignore')

# ── Load model ──
try:
    MODEL     = joblib.load('icu_mortality_model.pkl')
    THRESHOLD = joblib.load('icu_mortality_threshold.pkl')
    THRESHOLD = float(THRESHOLD)
    MODEL_STATUS = f'Live model loaded  |  Threshold: {THRESHOLD:.4f}'
    MODEL_OK = True
except FileNotFoundError:
    MODEL, THRESHOLD, MODEL_OK = None, 0.143, False
    MODEL_STATUS = 'Model not found — running in demo mode'

# ── Load reference data ──
try:
    REF_DF = pd.read_csv('mimic dataset/icu_final_df.csv')
    REF_DF['outcome'] = REF_DF['mortality_icu'].map({0:'Survived',1:'Died'})
    REF_OK = True
    SUMMARY = None
except:
    REF_DF, REF_OK = None, False
    # ── Fallback: load pre-aggregated summary JSON ──
    try:
        with open('dashboard_summary.json') as f:
            SUMMARY = json.load(f)
        SUMMARY_OK = True
    except:
        SUMMARY, SUMMARY_OK = None, False

In [3]:
# ── Design tokens ──
C_NAVY    = '#0d2b4e'
C_BLUE    = '#1a5fa8'
C_BLUE_LT = '#e8f0fb'
C_ORANGE  = '#e86c1a'
C_ORG_LT  = '#fdf0e8'
C_BG      = '#f4f6f9'
C_WHITE   = '#ffffff'
C_TEXT    = '#1a1a2e'
C_MUTED   = '#5a6478'
C_BORDER  = '#d0d8e8'
C_GREEN   = '#2d8a5e'
C_RED     = '#c0392b'
C_AMBER   = '#c8780a'
PLOT_BG   = '#f9fafb'

NORMAL_RANGES = {
    'mean_hr':         (60,  100, 40,  140),
    'mean_sbp':        (90,  140, 70,  180),
    'mean_dbp':        (60,  90,  40,  120),
    'mean_map':        (70,  100, 50,  130),
    'mean_rr':         (12,  20,  8,   30),
    'mean_spo2':       (95,  100, 88,  100),
    'mean_temp_c':     (36.1,37.2,35.0,39.5),
    'mean_lactate':    (0.5, 2.0, 0,   4.0),
    'mean_creatinine': (0.6, 1.2, 0,   3.0),
    'mean_bilirubin':  (0.2, 1.2, 0,   5.0),
    'mean_wbc':        (4.5, 11.0,2.0, 20.0),
    'mean_hemog':      (12,  17,  7,   20),
    'mean_sodium':     (136, 145, 125, 155),
    'mean_potassium':  (3.5, 5.0, 2.5, 6.5),
    'mean_bun':        (7,   20,  0,   50),
    'mean_platelets':  (150, 400, 50,  700),
}

def get_alert_level(feat, val):
    if feat not in NORMAL_RANGES or val is None or (isinstance(val, float) and np.isnan(val)):
        return 'normal'
    lo_n, hi_n, lo_c, hi_c = NORMAL_RANGES[feat]
    if val < lo_c or val > hi_c: return 'critical'
    if val < lo_n or val > hi_n: return 'warning'
    return 'normal'

def plot_layout(h=320, legend=True):
    ax = dict(color=C_TEXT, tickcolor=C_TEXT,
              tickfont=dict(color=C_TEXT, size=11),
              title=dict(font=dict(color=C_TEXT, size=12)),
              linecolor=C_BORDER, gridcolor='#e8ecf2')
    d = dict(height=h, template='plotly_white',
             paper_bgcolor=PLOT_BG, plot_bgcolor=PLOT_BG,
             font=dict(family='Inter, sans-serif', color=C_TEXT, size=12),
             margin=dict(t=24, b=24, l=24, r=24),
             xaxis=ax, yaxis=ax)
    if legend:
        d['legend'] = dict(orientation='h', y=1.1,
                           font=dict(color=C_TEXT, size=12),
                           bgcolor='rgba(0,0,0,0)')
    else:
        d['showlegend'] = False
    return d



In [19]:
# ── Global CSS ──
display(HTML(f'''
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&family=Playfair+Display:wght@600&display=swap');

body, .jp-Notebook {{ background: linear-gradient(160deg, #0d2b4e 0%, #1a5276 40%, #1a5fa8 100%) fixed !important; min-height:100vh !important; font-family: Inter, sans-serif; }}
.jp-Cell, .jp-Cell-inputWrapper, .jp-Cell-outputWrapper {{ padding: 0 !important; margin: 0 !important; background: transparent !important; }}
.jp-OutputArea-output {{ padding: 0 !important; }}
.jp-RenderedHTMLCommon {{ padding: 0 !important; }}

.dash-header {{
    background: linear-gradient(135deg, {C_NAVY} 0%, {C_BLUE} 100%);
    padding: 24px 36px;
    border-radius: 0 0 16px 16px;
    margin-bottom: 0px;
    display: flex;
    align-items: center;
    justify-content: center;
    text-align: center;
}}
.dash-header h1 {{
    font-family: 'Playfair Display', serif;
    font-size: 1.65em;
    font-weight: 600;
    color: white;
    margin: 0 0 4px 0;
}}
.dash-header .sub {{ font-size: 0.82em; color: rgba(255,255,255,0.75); margin: 0; }}

.tab-bar {{
    display: flex;
    gap: 6px;
    padding: 0 8px 16px 8px;
    border-bottom: 2px solid {C_BORDER};
    margin-bottom: 24px;
}}
.tab-btn {{
    padding: 9px 22px;
    border: 1.5px solid {C_BORDER};
    border-radius: 8px;
    background: {C_WHITE};
    color: {C_MUTED};
    font-size: 0.88em;
    font-weight: 600;
    cursor: pointer;
    font-family: Inter, sans-serif;
    transition: all 0.18s;
    letter-spacing: 0.02em;
}}
.tab-btn:hover {{ background: {C_BLUE_LT}; color: {C_BLUE}; border-color: {C_BLUE}; }}
.tab-btn.active {{ background: {C_BLUE}; color: white; border-color: {C_BLUE}; }}

.jp-Notebook {{ max-width: 100%; padding: 0; }}
.jp-Cell {{ max-width: 100%; margin: 0 !important; }}

.card {{
    background: {C_WHITE};
    border-radius: 12px;
    padding: 22px 24px;
    margin-bottom: 20px;
    border: 1px solid {C_BORDER};
    box-shadow: 0 2px 8px rgba(13,43,78,0.06);
}}
.card-title {{
    font-size: 0.95em;
    font-weight: 700;
    color: {C_NAVY};
    text-transform: uppercase;
    letter-spacing: 0.06em;
    margin-bottom: 14px;
    padding-bottom: 10px;
    border-bottom: 2px solid {C_BLUE_LT};
}}

.stat-grid {{ display: flex; gap: 14px; margin-bottom: 22px; flex-wrap: wrap; }}
.stat-box {{
    flex: 1;
    min-width: 140px;
    background: {C_WHITE};
    border-radius: 12px;
    padding: 18px 16px;
    border: 1px solid {C_BORDER};
    box-shadow: 0 2px 8px rgba(13,43,78,0.05);
    text-align: center;
}}
.stat-box .sv {{ font-family: 'Playfair Display', serif; font-size: 2em; font-weight: 600; margin: 6px 0 4px 0; }}
.stat-box .sl {{ font-size: 0.72em; font-weight: 700; text-transform: uppercase; letter-spacing: 0.07em; color: {C_MUTED}; }}

.risk-banner {{
    border-radius: 12px;
    padding: 22px 28px;
    margin-bottom: 22px;
    border-left: 6px solid;
    border-top: 1px solid;
    border-right: 1px solid;
    border-bottom: 1px solid;
}}
.risk-banner h2 {{
    font-family: 'Playfair Display', serif;
    font-size: 1.6em;
    font-weight: 600;
    margin: 0 0 6px 0;
    color: {C_TEXT};
}}
.risk-banner p {{ margin: 0; font-size: 0.95em; color: {C_MUTED}; }}
.risk-banner b {{ color: {C_TEXT}; }}

.alert-box {{
    border-radius: 8px;
    padding: 12px 16px;
    margin: 8px 0;
    font-size: 0.88em;
    line-height: 1.7;
}}
.alert-crit {{ background: #fdf0ee; border-left: 4px solid {C_RED}; color: {C_TEXT}; }}
.alert-warn {{ background: #fdf6e8; border-left: 4px solid {C_AMBER}; color: {C_TEXT}; }}
.alert-ok   {{ background: #edf7f2; border-left: 4px solid {C_GREEN}; color: {C_TEXT}; }}
.alert-rec  {{ background: {C_BLUE_LT}; border-left: 4px solid {C_BLUE}; color: {C_TEXT}; }}
.alert-box b {{ color: {C_TEXT}; }}

.two-col {{ display: flex; gap: 20px; }}
.two-col > div {{ flex: 1; }}

.section-label {{
    font-size: 0.78em;
    font-weight: 700;
    text-transform: uppercase;
    letter-spacing: 0.08em;
    color: {C_ORANGE};
    margin: 20px 0 10px 0;
    padding-bottom: 4px;
    border-bottom: 1px solid {C_ORG_LT};
}}

.info-table {{ width: 100%; border-collapse: collapse; font-size: 0.9em; }}
.info-table tr {{ border-bottom: 1px solid {C_BLUE_LT}; }}
.info-table td {{ padding: 9px 6px; color: {C_TEXT}; }}
.info-table td:first-child {{ color: {C_MUTED}; font-weight: 600; width: 48%; }}

.footer {{
    text-align: center;
    font-size: 0.75em;
    color: {C_MUTED};
    padding: 16px;
    margin-top: 16px;
    border-top: 1px solid {C_BORDER};
}}
.jp-OutputArea-output pre {{ display: none !important; }}
</style>
'''))

In [5]:
# ── Header ──
display(HTML(f'''
<div class="dash-header">
  <div>
    <h1>ICU 72-Hour Mortality Risk Dashboard</h1>
    <p class="sub">Clinical decision support for healthcare providers and nurse practitioners</p>
  </div>  
</div>
'''))

In [36]:
common_layout = widgets.Layout(flex='1', height='48px')

tab_home     = widgets.Button(description='Home',          layout=common_layout)
tab_overview = widgets.Button(description='Overview',      layout=common_layout)
tab_data     = widgets.Button(description='Data Explorer', layout=common_layout)

all_tabs     = [tab_home, tab_overview, tab_data]
current_page = ['home']

for btn in all_tabs:
    btn.style.button_color = '#1a5fa8'
    btn.style.text_color   = 'white'
    btn.layout.border      = 'none'

# Make Home tab slightly different
tab_home.style.button_color = C_NAVY

page_output = widgets.Output()

display(widgets.HBox(all_tabs, layout=widgets.Layout(
    margin='0 0 0 0', width='100%'
)))
display(page_output)




Output()

In [ ]:
# ══════════════════════════════════════════════
# OVERVIEW FROM SUMMARY JSON (no raw data)
# ══════════════════════════════════════════════
def render_overview_from_summary():
    with page_output:
        clear_output(wait=True)

        if not SUMMARY_OK:
            display(HTML('<div class="card">Summary data not available.</div>'))
            return

        s          = SUMMARY
        n_total    = s['n_total']
        n_died     = s['n_died']
        n_survived = s['n_survived']
        mort_rate  = s['mort_rate']
        median_age = int(s['median_age'])
        median_los = round(s['median_los'], 1)

        # ── Stat boxes ──
        display(HTML(f'''
        <div class="stat-grid">
          <div class="stat-box" style="border-top:4px solid {C_BLUE}">
            <div class="sl">Total Patients</div>
            <div class="sv" style="color:{C_BLUE}">{n_total:,}</div>
          </div>
          <div class="stat-box" style="border-top:4px solid {C_GREEN}">
            <div class="sl">Survived</div>
            <div class="sv" style="color:{C_GREEN}">{n_survived:,}</div>
          </div>
          <div class="stat-box" style="border-top:4px solid {C_RED}">
            <div class="sl">Died in ICU</div>
            <div class="sv" style="color:{C_RED}">{n_died:,}</div>
          </div>
          <div class="stat-box" style="border-top:4px solid {C_ORANGE}">
            <div class="sl">Mortality Rate</div>
            <div class="sv" style="color:{C_ORANGE}">{mort_rate:.1%}</div>
          </div>
          <div class="stat-box" style="border-top:4px solid {C_NAVY}">
            <div class="sl">Median Age</div>
            <div class="sv" style="color:{C_NAVY}">{median_age} yrs</div>
          </div>
          <div class="stat-box" style="border-top:4px solid #8a6aa8">
            <div class="sl">Median LOS</div>
            <div class="sv" style="color:#8a6aa8">{median_los}d</div>
          </div>
        </div>
        '''))

        # ── Outcome pie ──
        # ── Outcome pie ──
        fig_pie = go.Figure(go.Pie(
            labels=['Survived', 'Died'],
            values=[n_survived, n_died],
            marker_colors=[C_BLUE, C_ORANGE],
            hole=0.52,
            textinfo='label+percent',
            textfont=dict(size=13, color=C_TEXT)
        ))
        fig_pie.update_layout(height=300, showlegend=False,
                              paper_bgcolor=PLOT_BG, plot_bgcolor=PLOT_BG,
                              margin=dict(t=48,b=24,l=24,r=24),
                              font=dict(family='Inter',color=C_TEXT),
                              title=dict(text='Outcome Distribution',
                                         font=dict(color=C_NAVY, size=13),
                                         x=0.5, xanchor='center',
                                         y=0.97, yanchor='top'))
        display(fig_pie)

        # ── Age histogram ──
        bins = s['age_bins']
        fig_age = go.Figure()
        fig_age.add_trace(go.Bar(
            x=bins, y=s['age_hist_survived'],
            name='Survived', marker_color=C_BLUE, opacity=0.75
        ))
        fig_age.add_trace(go.Bar(
            x=bins, y=s['age_hist_died'],
            name='Died', marker_color=C_ORANGE, opacity=0.75
        ))
        fig_age.update_layout(barmode='overlay', height=300,
                              paper_bgcolor=PLOT_BG, plot_bgcolor=PLOT_BG,
                              margin=dict(t=48,b=24,l=24,r=24),
                              font=dict(family='Inter',color=C_TEXT),
                              xaxis_title='Age', yaxis_title='Count',
                              legend=dict(orientation='h', y=1.12,
                                          xanchor='center', x=0.5,
                                          font=dict(color=C_TEXT,size=12),
                                          bgcolor='rgba(0,0,0,0)'),
                              title=dict(text='Age Distribution by Outcome',
                                         font=dict(color=C_NAVY, size=13),
                                         x=0.5, xanchor='center',
                                         y=0.97, yanchor='top'))
        display(fig_age)

        # ── Care unit mortality ──
        unit_mort  = s['unit_mort']
        unit_names = list(unit_mort.keys())
        unit_vals  = [unit_mort[u]*100 for u in unit_names]
        sorted_pairs = sorted(zip(unit_vals, unit_names))
        unit_vals, unit_names = zip(*sorted_pairs)

        fig_unit = go.Figure(go.Bar(
            x=list(unit_vals), y=list(unit_names),
            orientation='h',
            marker=dict(color=list(unit_vals),
                        colorscale=[[0, C_BLUE],[1, C_ORANGE]]),
            text=[f'{v:.1f}%' for v in unit_vals],
            textposition='outside',
            textfont=dict(color=C_TEXT)
        ))
        fig_unit.update_layout(height=320,
                               paper_bgcolor=PLOT_BG, plot_bgcolor=PLOT_BG,
                               margin=dict(t=48,b=24,l=24,r=24),
                               showlegend=False,
                               font=dict(family='Inter',color=C_TEXT),
                               xaxis_title='Mortality %',
                               title=dict(text='Mortality Rate by Care Unit',
                                          font=dict(color=C_NAVY, size=13),
                                          x=0.5, xanchor='center',
                                          y=0.97, yanchor='top'))
        display(fig_unit)

        # ── Lab medians ──
        lab_map = {
            'mean_lactate':'Lactate','mean_creatinine':'Creatinine',
            'mean_bun':'BUN','mean_bilirubin':'Bilirubin','mean_wbc':'WBC'
        }
        lab_labels, surv_vals, died_vals = [], [], []
        for k, label in lab_map.items():
            if k in s['lab_medians']:
                lab_labels.append(label)
                surv_vals.append(s['lab_medians'][k]['survived'])
                died_vals.append(s['lab_medians'][k]['died'])

        fig_lab = go.Figure()
        fig_lab.add_trace(go.Bar(name='Survived', x=lab_labels,
                                  y=surv_vals, marker_color=C_BLUE, opacity=0.9))
        fig_lab.add_trace(go.Bar(name='Died', x=lab_labels,
                                  y=died_vals, marker_color=C_ORANGE, opacity=0.9))
        fig_lab.update_layout(barmode='group', height=320,
                               bargap=0.3, bargroupgap=0.15,
                               paper_bgcolor=PLOT_BG, plot_bgcolor=PLOT_BG,
                               margin=dict(t=48,b=24,l=24,r=24),
                               font=dict(family='Inter',color=C_TEXT),
                               legend=dict(orientation='h', y=1.12,
                                           font=dict(color=C_TEXT,size=12),
                                           bgcolor='rgba(0,0,0,0)'),
                               title=dict(text='Key Lab Medians by Outcome',
                                          font=dict(color=C_NAVY, size=13),
                                          x=0.5, xanchor='center',
                                          y=0.97, yanchor='top'))
        display(fig_lab)

        # ── LOS box ──
        fig_los = go.Figure()
        for label, color, key in [('Survived', C_BLUE, 'los_survived'),
                                    ('Died',     C_ORANGE, 'los_died')]:
            d = s[key]
            fig_los.add_trace(go.Box(
                name=label,
                x=[label],
                q1=[d['q25']],
                median=[d['median']],
                q3=[d['q75']],
                mean=[d['mean']],
                lowerfence=[d['min']],
                upperfence=[d['max']],
                marker_color=color,
                boxmean=True,
                boxpoints=False,
                width=0.4
            ))
        fig_los.update_layout(
            height=320,
            paper_bgcolor=PLOT_BG, plot_bgcolor=PLOT_BG,
            margin=dict(t=48,b=24,l=24,r=24),
            font=dict(family='Inter', color=C_TEXT),
            yaxis_title='LOS (days)',
            legend=dict(orientation='h', y=1.12,
                        font=dict(color=C_TEXT, size=12),
                        bgcolor='rgba(0,0,0,0)'),
            title=dict(text='Length of Stay by Outcome',
                       font=dict(color=C_NAVY, size=13),
                       x=0.5, xanchor='center',
                       y=0.97, yanchor='top')
        )
        fig_los.update_xaxes(range=[-1, 2])
        display(fig_los)


        display(HTML(f'<div class="footer">ICU Mortality Risk Dashboard &nbsp;|&nbsp; For clinical decision support only</div>'))

In [ ]:
# ══════════════════════════════════════════════
# OVERVIEW PAGE
# ══════════════════════════════════════════════
def render_overview():
    with page_output:
        clear_output(wait=True)
        if not REF_OK:
            if SUMMARY_OK:
                render_overview_from_summary()
            else:
                display(HTML('<div class="card">Reference dataset not available.</div>'))
            return

        df         = REF_DF
        n_total    = len(df)
        n_died     = int(df['mortality_icu'].sum())
        n_survived = n_total - n_died
        mort_rate  = n_died / n_total
        median_age = int(df['age'].median())
        median_los = df['los'].median()

        # Stat boxes
        display(HTML(f'''
        <div class="stat-grid">
          <div class="stat-box" style="border-top:4px solid {C_BLUE}">
            <div class="sl">Total Patients</div>
            <div class="sv" style="color:{C_BLUE}">{n_total:,}</div>
          </div>
          <div class="stat-box" style="border-top:4px solid {C_GREEN}">
            <div class="sl">Survived</div>
            <div class="sv" style="color:{C_GREEN}">{n_survived:,}</div>
          </div>
          <div class="stat-box" style="border-top:4px solid {C_RED}">
            <div class="sl">Died in ICU</div>
            <div class="sv" style="color:{C_RED}">{n_died:,}</div>
          </div>
          <div class="stat-box" style="border-top:4px solid {C_ORANGE}">
            <div class="sl">Mortality Rate</div>
            <div class="sv" style="color:{C_ORANGE}">{mort_rate:.1%}</div>
          </div>
          <div class="stat-box" style="border-top:4px solid {C_NAVY}">
            <div class="sl">Median Age</div>
            <div class="sv" style="color:{C_NAVY}">{median_age} yrs</div>
          </div>
          <div class="stat-box" style="border-top:4px solid #8a6aa8">
            <div class="sl">Median LOS</div>
            <div class="sv" style="color:#8a6aa8">{median_los:.1f}d</div>
          </div>
        </div>
        '''))

        # Row 1: pie + age
        fig_pie = go.Figure(go.Pie(
            labels=['Survived','Died'], values=[n_survived,n_died],
            marker_colors=[C_BLUE, C_ORANGE], hole=0.52,
            textinfo='label+percent', textfont=dict(size=13,color=C_TEXT)
        ))
        fig_pie.update_layout(**plot_layout(300, legend=False),
                              title=dict(text='Outcome Distribution',font=dict(color=C_NAVY,size=13)))

        fig_age = go.Figure()
        for outcome,color in [('Survived',C_BLUE),('Died',C_ORANGE)]:
            fig_age.add_trace(go.Histogram(
                x=df[df['outcome']==outcome]['age'], name=outcome,
                marker_color=color, opacity=0.75, nbinsx=30
            ))
        fig_age.update_layout(barmode='overlay', **plot_layout(300),
                              title=dict(text='Age Distribution by Outcome',font=dict(color=C_NAVY,size=13)))

        display(widgets.HBox([
            widgets.Output(),
            widgets.Output()
        ]))
        # Use plain plotly show
        display(fig_pie)
        display(fig_age)

        # Row 2: care unit + labs
        unit_mort = (df.groupby('careunit')['mortality_icu']
                       .mean().sort_values()*100).reset_index()
        fig_unit = px.bar(unit_mort, x='mortality_icu', y='careunit', orientation='h',
                          color='mortality_icu',
                          color_continuous_scale=[C_BLUE, C_ORANGE],
                          labels={'mortality_icu':'Mortality %','careunit':''})
        fig_unit.update_layout(**plot_layout(320, legend=False), coloraxis_showscale=False,
                               title=dict(text='Mortality Rate by Care Unit',font=dict(color=C_NAVY,size=13)))
        fig_unit.update_traces(texttemplate='%{x:.1f}%', textposition='outside',
                               textfont=dict(color=C_TEXT))

        labs       = ['mean_lactate','mean_creatinine','mean_bun','mean_bilirubin','mean_wbc']
        lab_labels = ['Lactate','Creatinine','BUN','Bilirubin','WBC']
        fig_lab = go.Figure()
        for outcome,color in [('Survived',C_BLUE),('Died',C_ORANGE)]:
            sub = df[df['outcome']==outcome]
            fig_lab.add_trace(go.Bar(
                name=outcome, x=lab_labels,
                y=[sub[c].median() for c in labs],
                marker_color=color, opacity=0.9
            ))
        fig_lab.update_layout(barmode='group', **plot_layout(320),
                              title=dict(text='Key Lab Medians by Outcome',font=dict(color=C_NAVY,size=13)))

        display(fig_unit)
        display(fig_lab)

        # Row 3: LOS box
        fig_los = go.Figure()
        for outcome,color in [('Survived',C_BLUE),('Died',C_ORANGE)]:
            sub = df[df['outcome']==outcome]['los'].dropna()
            sub = sub[sub < sub.quantile(0.97)]
            fig_los.add_trace(go.Box(
                y=sub, name=outcome, marker_color=color,
                boxmean=True, boxpoints=False
            ))
        fig_los.update_layout(**plot_layout(320), yaxis_title='LOS (days)',
                              title=dict(text='Length of Stay by Outcome',font=dict(color=C_NAVY,size=13)))
        display(fig_los)

        display(HTML(f'<div class="footer">ICU Mortality Risk Dashboard &nbsp;|&nbsp; For clinical decision support only</div>'))

In [33]:
# ══════════════════════════════════════════════
# PATIENT ASSESSMENT PAGE
# ══════════════════════════════════════════════
W  = {'layout': widgets.Layout(width='420px'), 'style': {'description_width': '140px'}}
WN = {'layout': widgets.Layout(width='420px'), 'style': {'description_width': '140px'}}

def inp(desc, val=''):
    # auto-size description width based on label length
    dw = '140px' if len(desc) > 12 else '110px'
    return widgets.Text(value=str(val), description=desc,
                        placeholder='Value or NA',
                        layout=widgets.Layout(width='420px'),
                        style={'description_width': dw})

def parse_val(w):
    v = str(w.value).strip()
    if v.upper() in ('NA', 'N/A', '', 'NONE', '-'):
        return float('nan')
    try:
        return float(v)
    except:
        return float('nan')

def parse_lab(w):
    return parse_val(w)

def lab_input(desc, val=''):
    return widgets.Text(value=str(val), description=desc,
                        placeholder='Value or NA',
                        layout=widgets.Layout(width='380px'),
                        style={'description_width': '110px'})

# Demographics
w_age      = inp('Age (years)', '65')
w_gender = widgets.Select(
                options=['Male', 'Female'],
                value='Male',
                rows=1,
                description='Gender',
                layout=widgets.Layout(width='380px', height='32px'),
                style={'description_width':'110px'})
w_race     = widgets.Dropdown(
    options=['WHITE','BLACK/AFRICAN AMERICAN','HISPANIC/LATINO','ASIAN','OTHER/UNKNOWN'],
    description='Race', **W)
w_careunit = widgets.Dropdown(options=[
    'Medical Intensive Care Unit (MICU)', 'Surgical Intensive Care Unit (SICU)',
    'Cardiac Vascular Intensive Care Unit (CVICU)',
    'Medical/Surgical Intensive Care Unit (MICU/SICU)',
    'Coronary Care Unit (CCU)', 'Neuro Surgical Intensive Care Unit (Neuro SICU)',
    'Trauma SICU (TSICU)'], description='Care Unit', **W)
w_admit    = widgets.Dropdown(
    options=['EMERGENCY ROOM','TRANSFER FROM HOSPITAL','PHYSICIAN REFERRAL',
             'WALK-IN/SELF REFERRAL','PROCEDURE SITE','PACU'],
    description='Admission Via', **W)
w_los      = inp('LOS so far (days)', '3')


# Vitals
w_hr=inp('Mean HR','');      w_hr_min=inp('Min HR','');      w_hr_max=inp('Max HR','')
w_sbp=inp('Mean SBP','');   w_sbp_min=inp('Min SBP','');    w_sbp_max=inp('Max SBP','')
w_dbp=inp('Mean DBP','');   w_dbp_min=inp('Min DBP','');    w_dbp_max=inp('Max DBP','')
w_map=inp('Mean MAP','');   w_map_min=inp('Min MAP','');    w_map_max=inp('Max MAP','')
w_rr=inp('Mean RR','');     w_rr_min=inp('Min RR','');      w_rr_max=inp('Max RR','')
w_spo2=inp('Mean SpO2','');    w_spo2_min=inp('Min SpO2','');  w_spo2_max=inp('Max SpO2','')
w_temp=inp('Mean Temp','');    w_fio2=inp('Max FiO2','')
w_pao2=inp('Max PaO2','')
w_urine=inp('Urine 72h (mL)','')
w_urine.layout.width = '1160px'

# Labs
w_cr=lab_input('Mean Creatinine','');   w_cr_min=lab_input('Min Creatinine','');   w_cr_max=lab_input('Max Creatinine','')
w_lac=lab_input('Mean Lactate','');     w_lac_min=lab_input('Min Lactate','');     w_lac_max=lab_input('Max Lactate','')
w_bili=lab_input('Mean Bilirubin','');  w_bili_min=lab_input('Min Bilirubin','');  w_bili_max=lab_input('Max Bilirubin','')
w_wbc=lab_input('Mean WBC',''); w_wbc_min=lab_input('Min WBC','');         w_wbc_max=lab_input('Max WBC','')
w_hgb=lab_input('Mean Hemoglobin',''); w_hgb_min=lab_input('Min Hemoglobin','');  w_hgb_max=lab_input('Max Hemoglobin','')
w_plt=lab_input('Mean Platelets','');  w_plt_min=lab_input('Min Platelets','');   w_plt_max=lab_input('Max Platelets','')
w_na=lab_input('Mean Sodium','');      w_na_min=lab_input('Min Sodium','');       w_na_max=lab_input('Max Sodium','')
w_k=lab_input('Mean Potassium','');    w_k_min=lab_input('Min Potassium','');     w_k_max=lab_input('Max Potassium','')
w_bun=lab_input('Mean BUN','');        w_bun_min=lab_input('Min BUN','');         w_bun_max=lab_input('Max BUN','')

assess_btn = widgets.Button(
    description='Assess Mortality Risk',
    layout=widgets.Layout(width='260px', height='44px')
)
assess_btn.style.button_color = C_ORANGE
assess_btn.style.text_color   = 'white'

result_out = widgets.Output()

def build_patient_df():
    return pd.DataFrame([{
        'age':parse_val(w_age), 'gender':w_gender.value, 'los':parse_val(w_los),
        'max_hr':parse_val(w_hr_max),'min_hr':parse_val(w_hr_min),'mean_hr':parse_val(w_hr),
        'max_sbp':parse_val(w_sbp_max),'min_sbp':parse_val(w_sbp_min),'mean_sbp':parse_val(w_sbp),
        'max_dbp':parse_val(w_dbp_max),'min_dbp':parse_val(w_dbp_min),'mean_dbp':parse_val(w_dbp),
        'min_map':parse_val(w_map_min),'max_map':parse_val(w_map_max),'mean_map':parse_val(w_map),
        'max_rr':parse_val(w_rr_max),'min_rr':parse_val(w_rr_min),'mean_rr':parse_val(w_rr),
        'min_spo2':parse_val(w_spo2_min),'max_spo2':parse_val(w_spo2_max),'mean_spo2':parse_val(w_spo2),
        'max_fio2':parse_val(w_fio2),'max_pao2':parse_val(w_pao2),'mean_temp_c':parse_val(w_temp),
        'total_urine_72h':parse_val(w_urine),
        'max_creatinine':parse_lab(w_cr_max),'min_creatinine':parse_lab(w_cr_min),'mean_creatinine':parse_lab(w_cr),
        'max_lactate':parse_lab(w_lac_max),'min_lactate':parse_lab(w_lac_min),'mean_lactate':parse_lab(w_lac),
        'max_bilirubin':parse_lab(w_bili_max),'min_bilirubin':parse_lab(w_bili_min),'mean_bilirubin':parse_lab(w_bili),
        'max_wbc':parse_lab(w_wbc_max),'min_wbc':parse_lab(w_wbc_min),'mean_wbc':parse_lab(w_wbc),
        'max_hemog':parse_lab(w_hgb_max),'min_hemog':parse_lab(w_hgb_min),'mean_hemog':parse_lab(w_hgb),
        'max_platelets':parse_lab(w_plt_max),'min_platelets':parse_lab(w_plt_min),'mean_platelets':parse_lab(w_plt),
        'max_sodium':parse_lab(w_na_max),'min_sodium':parse_lab(w_na_min),'mean_sodium':parse_lab(w_na),
        'max_potassium':parse_lab(w_k_max),'min_potassium':parse_lab(w_k_min),'mean_potassium':parse_lab(w_k),
        'max_bun':parse_lab(w_bun_max),'min_bun':parse_lab(w_bun_min),'mean_bun':parse_lab(w_bun),
        'race_clean':w_race.value,'careunit':w_careunit.value,'admission_loc':w_admit.value,
    }])

def on_assess(b):
    with result_out:
        clear_output(wait=True)

          # ── Validation — require at least age + 3 vitals ──
        required = {
            'Age':       parse_val(w_age),
            'Mean HR':   parse_val(w_hr),
            'Mean SBP':  parse_val(w_sbp),
            'Mean MAP':  parse_val(w_map),
            'Mean SpO2': parse_val(w_spo2),
            'Mean RR':   parse_val(w_rr),
        }
        missing = [k for k,v in required.items() if isinstance(v, float) and np.isnan(v)]

        if len(missing) > 3:
            display(HTML(f'''
            <div style="background:#fdf0ee;border-left:4px solid {C_RED};
                        border-radius:8px;padding:16px 20px;font-size:0.92em;color:{C_TEXT}">
                <b style="color:{C_RED}">Insufficient Data</b><br><br>
                Please enter at least <b>Age</b> and the core vitals
                (<b>HR, SBP, MAP, SpO2, RR</b>) before running the assessment.
                The following required fields are missing:
                <br><br>
                {''.join(f'&bull; {m}<br>' for m in missing)}
            </div>
            '''))
            return

        patient_df = build_patient_df()

        if MODEL is not None:
            prob = float(MODEL.predict_proba(patient_df)[0, 1])
        else:
            lac_v  = parse_lab(w_lac);  lac_v  = 1.0 if (isinstance(lac_v,float) and np.isnan(lac_v)) else lac_v
            cr_v   = parse_lab(w_cr);   cr_v   = 1.0 if (isinstance(cr_v,float)  and np.isnan(cr_v))  else cr_v
            map_v  = parse_val(w_map);  map_v  = 90  if (isinstance(map_v,float) and np.isnan(map_v)) else map_v
            spo2_v = parse_val(w_spo2); spo2_v = 97  if (isinstance(spo2_v,float) and np.isnan(spo2_v)) else spo2_v
            age_v  = parse_val(w_age);  age_v  = 65  if (isinstance(age_v,float) and np.isnan(age_v))  else age_v
            prob = min(0.99, max(0.01,
                0.05 + (age_v-50)*0.004
                + max(0,lac_v-2.0)*0.08
                + max(0,cr_v-1.5)*0.05
                + max(0,90-spo2_v)*0.02
                + max(0,65-map_v)*0.01))

        prediction = 1 if prob >= THRESHOLD else 0

        if   prob < THRESHOLD:            rl,rb,rbo = 'LOW RISK',      '#edf7f2', C_GREEN
        elif prob < THRESHOLD + 0.15:     rl,rb,rbo = 'MODERATE RISK', '#fdf6e8', C_AMBER
        elif prob < THRESHOLD + 0.35:     rl,rb,rbo = 'HIGH RISK',     '#fdf0ee', C_ORANGE
        else:                             rl,rb,rbo = 'CRITICAL RISK', '#fbe8e8', C_RED

        key_v = {
            'mean_hr':parse_val(w_hr),'mean_sbp':parse_val(w_sbp),'mean_map':parse_val(w_map),
            'mean_rr':parse_val(w_rr),'mean_spo2':parse_val(w_spo2),'mean_temp_c':parse_val(w_temp),
            'mean_lactate':parse_lab(w_lac),'mean_creatinine':parse_lab(w_cr),
            'mean_bilirubin':parse_lab(w_bili),'mean_wbc':parse_lab(w_wbc),
            'mean_potassium':parse_lab(w_k),'mean_sodium':parse_lab(w_na),
            'mean_platelets':parse_lab(w_plt),'mean_bun':parse_lab(w_bun),
            'mean_hemog':parse_lab(w_hgb),
        }
        crit_flags, warn_flags = [], []
        for feat, val in key_v.items():
            if val is None or (isinstance(val,float) and np.isnan(val)): continue
            level = get_alert_level(feat, val)
            lbl   = feat.replace('mean_','').replace('_',' ').title()
            if feat in NORMAL_RANGES:
                lo_n,hi_n,lo_c,hi_c = NORMAL_RANGES[feat]
                if level == 'critical':
                    crit_flags.append(f"{lbl}: {val:.1f} &nbsp; [{'LOW' if val<lo_c else 'HIGH'}]")
                elif level == 'warning':
                    warn_flags.append(f"{lbl}: {val:.1f} &nbsp; [{'LOW' if val<lo_n else 'HIGH'}]")

        recs = []
        lac_v = parse_lab(w_lac); cr_v  = parse_lab(w_cr)
        plt_v = parse_lab(w_plt); k_v   = parse_lab(w_k)
        hgb_v = parse_lab(w_hgb); bun_v = parse_lab(w_bun)
        map_v = parse_val(w_map); spo2_v = parse_val(w_spo2)
        rr_v  = parse_val(w_rr);  urine_v = parse_val(w_urine)
        if not (isinstance(lac_v,float) and np.isnan(lac_v)) and lac_v > 2.0:
            recs.append('Elevated lactate &mdash; IV fluids, sepsis workup, source control')
        if not (isinstance(map_v,float) and np.isnan(map_v)) and map_v < 65:
            recs.append('MAP &lt; 65 mmHg &mdash; initiate vasopressor therapy per protocol')
        if not (isinstance(spo2_v,float) and np.isnan(spo2_v)) and spo2_v < 92:
            recs.append('SpO2 &lt; 92% &mdash; increase FiO2, evaluate for intubation')
        if not (isinstance(cr_v,float) and np.isnan(cr_v)) and cr_v > 2.0:
            recs.append('Elevated creatinine &mdash; nephrology consult, review nephrotoxins')
        if not (isinstance(rr_v,float) and np.isnan(rr_v)) and rr_v > 25:
            recs.append('Tachypnoea RR &gt; 25 &mdash; ABG, CXR, consider NIV/intubation')
        if not (isinstance(plt_v,float) and np.isnan(plt_v)) and plt_v < 100:
            recs.append('Thrombocytopenia &mdash; bleeding risk, review anticoagulation')
        if not (isinstance(k_v,float) and np.isnan(k_v)) and k_v > 5.5:
            recs.append('Hyperkalaemia &gt; 5.5 &mdash; ECG, calcium gluconate, kayexalate/dialysis')
        if not (isinstance(k_v,float) and np.isnan(k_v)) and k_v < 3.0:
            recs.append('Hypokalaemia &lt; 3.0 &mdash; IV potassium replacement, cardiac monitoring')
        if not (isinstance(hgb_v,float) and np.isnan(hgb_v)) and hgb_v < 7:
            recs.append('Hgb &lt; 7 g/dL &mdash; consider transfusion per protocol')
        if not (isinstance(bun_v,float) and np.isnan(bun_v)) and bun_v > 40:
            recs.append('Elevated BUN &mdash; assess for AKI, GI bleed, volume status')
        if not (isinstance(urine_v,float) and np.isnan(urine_v)) and urine_v < 500:
            recs.append('Oliguria &lt; 500 mL/72h &mdash; fluid challenge, assess for AKI/obstruction')
        if prediction == 1 and not recs:
            recs.append('High composite risk &mdash; escalate monitoring, ICU team review')

        action = 'Intervention recommended' if prediction==1 else 'Continue standard monitoring'
        display(HTML(f'''
        <div class="risk-banner" style="background:{rb};border-color:{rbo}">
          <h2 style="color:{rbo}">{rl}</h2>
          <p>72h Mortality Probability: <b>{prob*100:.1f}%</b>
             &nbsp;|&nbsp; Threshold: <b>{THRESHOLD:.3f}</b>
             &nbsp;|&nbsp; {action}</p>
        </div>
        '''))

        t = THRESHOLD
        fig_g = go.Figure(go.Indicator(
            mode='gauge+number',
            value=round(prob*100,1),
            number={'suffix':'%','font':{'size':46,'color':C_TEXT}},
            gauge={
                'axis':{'range':[0,100],'tickcolor':C_MUTED,
                        'tickfont':{'color':C_MUTED,'size':11}},
                'bar':{'color':rbo,'thickness':0.26},
                'bgcolor':PLOT_BG, 'borderwidth':0,
                'steps':[
                    {'range':[0,t*100],                'color':'#dff0ea'},
                    {'range':[t*100,(t+.15)*100],       'color':'#fef3d8'},
                    {'range':[(t+.15)*100,(t+.35)*100], 'color':'#fde8d8'},
                    {'range':[(t+.35)*100,100],         'color':'#fad8d8'},
                ],
                'threshold':{'line':{'color':C_NAVY,'width':3},
                             'thickness':0.85,'value':t*100}
            }
        ))
        fig_g.update_layout(height=280, margin=dict(t=30,b=10,l=30,r=30),
                            paper_bgcolor=PLOT_BG,
                            font=dict(family='Inter',color=C_TEXT))
        display(fig_g)

        bar_labels = ['HR\n(bpm)','SBP\n(mmHg)','MAP\n(mmHg)','RR\n(/min)',
                      'SpO2\n(%)','Temp\n(C)','Lactate','Creatinine',
                      'K+\n(mEq/L)','Hgb\n(g/dL)','Platelets\n(K/uL)']
        bar_vals  = [parse_val(w_hr),parse_val(w_sbp),parse_val(w_map),parse_val(w_rr),
                     parse_val(w_spo2),parse_val(w_temp),
                     parse_lab(w_lac),parse_lab(w_cr),
                     parse_lab(w_k),parse_lab(w_hgb),parse_lab(w_plt)]
        bar_feats = ['mean_hr','mean_sbp','mean_map','mean_rr','mean_spo2',
                     'mean_temp_c','mean_lactate','mean_creatinine',
                     'mean_potassium','mean_hemog','mean_platelets']
        clr_map    = {'normal':C_BLUE,'warning':C_AMBER,'critical':C_RED}
        bar_colors = [clr_map[get_alert_level(f,v)] if not (isinstance(v,float) and np.isnan(v))
                      else '#cccccc' for f,v in zip(bar_feats,bar_vals)]
        bar_text   = [f'{v:.1f}' if not (isinstance(v,float) and np.isnan(v)) else 'NA'
                      for v in bar_vals]
        fig_bar = go.Figure(go.Bar(
            x=bar_labels,
            y=[v if not (isinstance(v,float) and np.isnan(v)) else 0 for v in bar_vals],
            marker_color=bar_colors,
            text=bar_text, textposition='outside',
            textfont=dict(color=C_TEXT, size=11)
        ))
        fig_bar.update_layout(**plot_layout(320, legend=False), yaxis_title='Value',
                              title=dict(
                                  text='Vitals and Labs  |  Blue=Normal  Orange=Warning  Red=Critical  Grey=NA',
                                  font=dict(size=11,color=C_MUTED)))
        display(fig_bar)

        crit_html = '<br>'.join(f'&bull; {f}' for f in crit_flags) if crit_flags else ''
        warn_html = '<br>'.join(f'&bull; {f}' for f in warn_flags) if warn_flags else ''
        recs_html = '<br>'.join(f'&bull; {r}' for r in recs) if recs else ''

        display(HTML(f'''
        <div class="two-col">
          <div class="card">
            <div class="card-title">Clinical Alerts</div>
            {f'<div class="alert-box alert-crit"><b>CRITICAL VALUES</b><br>{crit_html}</div>' if crit_flags else ''}
            {f'<div class="alert-box alert-warn"><b>WARNING VALUES</b><br>{warn_html}</div>' if warn_flags else ''}
            {f'<div class="alert-box alert-ok">All monitored values within normal range.</div>'
             if not crit_flags and not warn_flags else ''}
          </div>
          <div class="card">
            <div class="card-title">Clinical Action Items</div>
            {f'<div class="alert-box alert-rec"><b>Recommended Actions</b><br>{recs_html}</div>' if recs else
             '<div class="alert-box alert-ok">No immediate actions flagged.</div>'}
          </div>
        </div>
        '''))

        display(HTML(f'<div class="footer">ICU Mortality Risk Dashboard &nbsp;|&nbsp; '
                     f'For clinical decision support only &mdash; not a substitute for clinical judgment.</div>'))

assess_btn.on_click(on_assess)

In [ ]:
# ══════════════════════════════════════════════
# DATA EXPLORER PAGE
# ══════════════════════════════════════════════
def render_data():
    with page_output:
        clear_output(wait=True)
        if not REF_OK:
            display(HTML(f'''
            <div style="background:{C_BLUE_LT};border-left:4px solid {C_BLUE};
                        border-radius:8px;padding:16px 20px;font-size:0.9em;color:{C_NAVY}">
                <b>Raw data not available</b> — the MIMIC-IV dataset is excluded from
                this repository due to the PhysioNet data use agreement.
                Interactive feature exploration requires local access to the dataset.
            </div>
            '''))
            return

        df = REF_DF
        NUM_COLS = [c for c in df.select_dtypes('number').columns if c != 'mortality_icu']
        VITALS   = ['mean_hr','mean_sbp','mean_dbp','mean_map','mean_rr','mean_spo2','mean_temp_c']
        LABS     = ['mean_creatinine','mean_lactate','mean_bilirubin','mean_wbc',
                    'mean_hemog','mean_sodium','mean_potassium','mean_bun','mean_platelets']

        display(HTML('<div class="card"><div class="card-title">Distribution Explorer</div>'))

        feat_dd = widgets.Dropdown(
            options=NUM_COLS,
            value='mean_lactate' if 'mean_lactate' in NUM_COLS else NUM_COLS[0],
            description='Feature:',
            layout=widgets.Layout(width='320px'),
            style={'description_width':'80px'}
        )
        ptype_rb = widgets.RadioButtons(
            options=['Histogram','Box','Violin'],
            description='Plot type:',
            style={'description_width':'80px'},
            layout=widgets.Layout(width='300px')
        )
        dist_out = widgets.Output()

        def update_dist(change=None):
            with dist_out:
                clear_output(wait=True)
                feat  = feat_dd.value
                ptype = ptype_rb.value
                data  = df[[feat,'outcome']].dropna()
                kw = dict(color='outcome',
                          color_discrete_map={'Survived':C_BLUE,'Died':C_ORANGE})
                if ptype == 'Histogram':
                    fig = px.histogram(data, x=feat, barmode='overlay',
                                       opacity=0.75, nbins=60, **kw)
                elif ptype == 'Box':
                    fig = px.box(data, x='outcome', y=feat, boxmode='group', **kw)
                else:
                    fig = px.violin(data, x='outcome', y=feat, box=True, **kw)
                fig.update_layout(**plot_layout(380))
                display(fig)
                s_med = data[data['outcome']=='Survived'][feat].median()
                d_med = data[data['outcome']=='Died'][feat].median()
                display(HTML(f'<p style="color:{C_MUTED};font-size:0.88em">'
                             f'Survived median: <b style="color:{C_TEXT}">{s_med:.2f}</b>'
                             f' &nbsp;|&nbsp; '
                             f'Died median: <b style="color:{C_TEXT}">{d_med:.2f}</b></p>'))

        feat_dd.observe(update_dist, names='value')
        ptype_rb.observe(update_dist, names='value')
        display(widgets.HBox([feat_dd, ptype_rb]))
        display(dist_out)
        update_dist()
        display(HTML('</div>'))

        # Correlation heatmap
        display(HTML('<div class="card"><div class="card-title">Correlation Heatmap</div>'))
        grp_rb  = widgets.RadioButtons(
            options=['Vitals','Labs','All means'],
            value='Labs',
            description='Group:',
            style={'description_width':'60px'},
            layout=widgets.Layout(width='340px')
        )
        corr_out = widgets.Output()

        def update_corr(change=None):
            with corr_out:
                clear_output(wait=True)
                g = grp_rb.value
                if g == 'Vitals':    hcols = VITALS + ['mortality_icu']
                elif g == 'Labs':    hcols = LABS   + ['mortality_icu']
                else:                hcols = VITALS + LABS + ['age','los','mortality_icu']
                hcols = [c for c in hcols if c in df.columns]
                corr  = df[hcols].corr()
                fig = px.imshow(corr, text_auto='.2f', aspect='auto',
                                color_continuous_scale='RdBu_r', zmin=-1, zmax=1)
                fig.update_layout(**plot_layout(520, legend=False))
                fig.update_traces(textfont=dict(color=C_TEXT, size=10))
                display(fig)

        grp_rb.observe(update_corr, names='value')
        display(grp_rb)
        display(corr_out)
        update_corr()
        display(HTML('</div>'))

        # Missing data
        display(HTML('<div class="card"><div class="card-title">Missing Data Map</div>'))
        miss = (df.isnull().mean()*100).sort_values(ascending=False)
        miss = miss[miss>0].reset_index()
        miss.columns = ['feature','missing_pct']
        fig_m = px.bar(miss, x='missing_pct', y='feature', orientation='h',
                       color='missing_pct',
                       color_continuous_scale=[C_BLUE, C_ORANGE],
                       labels={'missing_pct':'Missing %'})
        lm = plot_layout(420, legend=False)
        lm['yaxis']['categoryorder'] = 'total ascending'
        lm['coloraxis_showscale'] = False
        fig_m.update_layout(**lm)
        display(fig_m)
        display(HTML('</div>'))

        display(HTML(f'<div class="footer">ICU Mortality Risk Dashboard &nbsp;|&nbsp; For clinical decision support only</div>'))



In [10]:
# ══════════════════════════════════════════════
# MODEL INFO PAGE
# ══════════════════════════════════════════════
def render_model_info():
    with page_output:
        clear_output(wait=True)

        ranges_rows = ''.join(
            f'<tr><td>{f.replace("mean_","").replace("_"," ").title()}</td>'
            f'<td>{lo_n}</td><td>{hi_n}</td><td>{lo_c}</td><td>{hi_c}</td></tr>'
            for f,(lo_n,hi_n,lo_c,hi_c) in NORMAL_RANGES.items()
        )

        display(HTML(f'''
        <div class="two-col">
          <div class="card">
            <div class="card-title">Model Configuration</div>
            <table class="info-table">
              <tr><td>Model type</td><td>Soft-voting ensemble</td></tr>
              <tr><td>Components</td><td>HGB (w=2) + RF (w=1) + XGBoost (w=2)</td></tr>
              <tr><td>Calibration</td><td>Platt sigmoid</td></tr>
              <tr><td>CV strategy</td><td>StratifiedKFold (5 fold)</td></tr>
              <tr><td>CV scorer</td><td>Recall with precision floor &ge; 0.40</td></tr>
              <tr><td>Resampling</td><td>SMOTE (k_neighbors=5)</td></tr>
              <tr><td>Decision threshold</td><td><b>{THRESHOLD:.4f}</b></td></tr>
              <tr><td>Mode</td><td style="color:{'#2d8a5e' if MODEL_OK else C_AMBER};font-weight:600">
                  {'Live model loaded' if MODEL_OK else 'Demo mode'}</td></tr>
            </table>
          </div>
          <div class="card">
            <div class="card-title">Class Imbalance Strategy</div>
            <div class="section-label">Resampling</div>
            <ul style="color:{C_TEXT};line-height:1.9;padding-left:18px;margin:0 0 8px 0">
              <li>SMOTE oversampling inside imblearn.Pipeline</li>
              <li>Applied only on training folds &mdash; no data leakage</li>
            </ul>
            <div class="section-label">Class Weighting</div>
            <ul style="color:{C_TEXT};line-height:1.9;padding-left:18px;margin:0 0 8px 0">
              <li>class_weight=balanced on HGB</li>
              <li>Explicit ratio search on RF (w in [5,7,10,15,20])</li>
              <li>scale_pos_weight = neg/pos on XGBoost</li>
            </ul>
            <div class="section-label">Threshold Selection</div>
            <ul style="color:{C_TEXT};line-height:1.9;padding-left:18px;margin:0">
              <li>Recall &ge; 0.80 constraint enforced first</li>
              <li>Precision floor &ge; 0.40 to avoid rubber-stamp predictions</li>
              <li>F1 maximised within those constraints</li>
            </ul>
          </div>
        </div>

        <div class="card">
          <div class="card-title">Clinical Normal Ranges Reference</div>
          <table style="width:100%;border-collapse:collapse;font-size:0.88em">
            <thead>
              <tr style="background:{C_BLUE_LT};color:{C_NAVY};font-weight:700">
                <td style="padding:9px 8px">Feature</td>
                <td style="padding:9px 8px">Normal Low</td>
                <td style="padding:9px 8px">Normal High</td>
                <td style="padding:9px 8px">Critical Low</td>
                <td style="padding:9px 8px">Critical High</td>
              </tr>
            </thead>
            <tbody style="color:{C_TEXT}">
              {ranges_rows}
            </tbody>
          </table>
        </div>

        <div class="footer">ICU Mortality Risk Dashboard &nbsp;|&nbsp; For clinical decision support only &mdash; not a substitute for clinical judgment.</div>
        '''))



In [11]:
# ══════════════════════════════════════════════
# ASSESSMENT PAGE LAYOUT
# ══════════════════════════════════════════════
def render_assessment():
    with page_output:
        clear_output(wait=True)

        display(HTML(f'''
        <div style="background:{C_BLUE_LT};border-left:4px solid {C_BLUE};
                    border-radius:8px;padding:12px 16px;margin-bottom:20px;
                    font-size:0.88em;color:{C_NAVY};line-height:1.7">
            <b>Prediction Window</b> — This model predicts <b>ICU mortality risk</b>
            using clinical data collected during the <b>first 72 hours</b> of ICU admission.If the patient has been in the ICU for less than 72 hours, enter values
            collected so far. Enter the mean, minimum, and maximum values observed over that window.
            The prediction does not indicate death within 72 hours, it estimates the
            overall probability of <b>in-ICU mortality</b> based on early clinical indicators.
            Leave unavailable fields blank or type <b>NA</b>, missing values will be
            automatically imputed.
        </div>
        '''))

        # ── Demographics ──
        display(HTML(f'''
        <div style="
            background:linear-gradient(135deg,#f8f9fb 0%,#f2f4f7 100%);
            border-radius:12px;padding:10px 16px;
            margin-top:8px;margin-bottom:12px;
            border:1px solid #dde2ea;border-left:4px solid #8a9ab5;
            box-shadow:0 2px 8px rgba(0,0,0,0.05);
        ">
            <div style="font-size:0.85em;font-weight:700;color:{C_NAVY};
                        text-transform:uppercase;letter-spacing:0.06em;
                        padding-bottom:6px;border-bottom:1px solid #dde2ea;">
                Patient Demographics and Admission
            </div>
        </div>
        '''))
        display(widgets.VBox([
            widgets.HBox([w_age, w_gender, w_los],
                        layout=widgets.Layout(gap='12px')),
            widgets.HBox([w_race, w_careunit, w_admit],
                        layout=widgets.Layout(gap='12px', margin='10px 0 0 0')),
        ], layout=widgets.Layout(margin='0 0 16px 0')))

        # ── Vital Signs ──
        display(HTML(f'''
        <div style="
            background:linear-gradient(135deg,#f8f9fb 0%,#f2f4f7 100%);
            border-radius:12px;padding:10px 16px;
            margin-top:24px;margin-bottom:12px;
            border:1px solid #dde2ea;border-left:4px solid #8a9ab5;
            box-shadow:0 2px 8px rgba(0,0,0,0.05);
        ">
            <div style="font-size:0.85em;font-weight:700;color:{C_NAVY};
                        text-transform:uppercase;letter-spacing:0.06em;
                        padding-bottom:6px;border-bottom:1px solid #dde2ea;">
                Vital Signs &nbsp;
                <span style="font-weight:400;font-size:0.9em;
                             color:{C_MUTED};text-transform:none">
                    Mean / Min / Max
                </span>
            </div>
        </div>
        '''))
        display(HTML(f'''
        <div style="
            background:white;
            border:1px solid #dde2ea;
            border-radius:10px;
            padding:14px 18px;
            margin-bottom:14px;
            font-size:0.82em;
        ">
            <div style="font-weight:700;color:{C_NAVY};margin-bottom:10px;
                        font-size:0.88em;text-transform:uppercase;letter-spacing:0.05em">
                Reference
            </div>
            <table style="width:100%;border-collapse:collapse;color:{C_MUTED}">
                <thead>
                    <tr style="background:#f4f6f9;font-size:0.8em;text-transform:uppercase;
                               letter-spacing:0.05em;color:{C_NAVY}">
                        <td style="padding:6px 8px;width:18%">Abbreviation</td>
                        <td style="padding:6px 8px;width:30%">Full Name</td>
                        <td style="padding:6px 8px;width:52%">Unit</td>
                    </tr>
                </thead>
                <tbody>
                    <tr style="border-bottom:1px solid #f0f2f5">
                        <td style="padding:5px 8px;font-weight:600;color:{C_TEXT}">HR</td>
                        <td style="padding:5px 8px">Heart Rate</td>
                        <td style="padding:5px 8px">beats per minute (bpm)</td>
                    </tr>
                    <tr style="border-bottom:1px solid #f0f2f5">
                        <td style="padding:5px 8px;font-weight:600;color:{C_TEXT}">SBP</td>
                        <td style="padding:5px 8px">Systolic Blood Pressure</td>
                        <td style="padding:5px 8px">mmHg</td>
                    </tr>
                    <tr style="border-bottom:1px solid #f0f2f5">
                        <td style="padding:5px 8px;font-weight:600;color:{C_TEXT}">DBP</td>
                        <td style="padding:5px 8px">Diastolic Blood Pressure</td>
                        <td style="padding:5px 8px">mmHg</td>
                    </tr>
                    <tr style="border-bottom:1px solid #f0f2f5">
                        <td style="padding:5px 8px;font-weight:600;color:{C_TEXT}">MAP</td>
                        <td style="padding:5px 8px">Mean Arterial Pressure</td>
                        <td style="padding:5px 8px">mmHg</td>
                    </tr>
                    <tr style="border-bottom:1px solid #f0f2f5">
                        <td style="padding:5px 8px;font-weight:600;color:{C_TEXT}">RR</td>
                        <td style="padding:5px 8px">Respiratory Rate</td>
                        <td style="padding:5px 8px">breaths per minute (/min)</td>
                    </tr>
                    <tr style="border-bottom:1px solid #f0f2f5">
                        <td style="padding:5px 8px;font-weight:600;color:{C_TEXT}">SpO2</td>
                        <td style="padding:5px 8px">Peripheral Oxygen Saturation</td>
                        <td style="padding:5px 8px">percentage (%)</td>
                    </tr>
                    <tr style="border-bottom:1px solid #f0f2f5">
                        <td style="padding:5px 8px;font-weight:600;color:{C_TEXT}">Temp</td>
                        <td style="padding:5px 8px">Body Temperature</td>
                        <td style="padding:5px 8px">degrees Celsius (&deg;C)</td>
                    </tr>
                    <tr style="border-bottom:1px solid #f0f2f5">
                        <td style="padding:5px 8px;font-weight:600;color:{C_TEXT}">FiO2</td>
                        <td style="padding:5px 8px">Fraction of Inspired Oxygen</td>
                        <td style="padding:5px 8px">fraction (0.21 &ndash; 1.0)</td>
                    </tr>
                    <tr>
                        <td style="padding:5px 8px;font-weight:600;color:{C_TEXT}">PaO2</td>
                        <td style="padding:5px 8px">Partial Pressure of Arterial Oxygen</td>
                        <td style="padding:5px 8px">mmHg</td>
                    </tr>
                </tbody>
            </table>
        </div>
        '''))
        display(widgets.VBox([
            widgets.HBox([w_hr,   w_hr_min,   w_hr_max]),
            widgets.HBox([w_sbp,  w_sbp_min,  w_sbp_max]),
            widgets.HBox([w_dbp,  w_dbp_min,  w_dbp_max]),
            widgets.HBox([w_map,  w_map_min,  w_map_max]),
            widgets.HBox([w_rr,   w_rr_min,   w_rr_max]),
            widgets.HBox([w_spo2, w_spo2_min, w_spo2_max]),
            widgets.HBox([w_temp, w_fio2,     w_pao2]),
        ], layout=widgets.Layout(margin='0 0 8px 0')))

        # ── Fluid Output ──
        display(HTML(f'''
        <div style="
            background:linear-gradient(135deg,#f8f9fb 0%,#f2f4f7 100%);
            border-radius:12px;padding:10px 16px;
            margin-top:24px;margin-bottom:12px;
            border:1px solid #dde2ea;border-left:4px solid #8a9ab5;
            box-shadow:0 2px 8px rgba(0,0,0,0.05);
        ">
            <div style="font-size:0.85em;font-weight:700;color:{C_NAVY};
                        text-transform:uppercase;letter-spacing:0.06em;
                        padding-bottom:6px;border-bottom:1px solid #dde2ea;">
                Fluid Output &nbsp;
            </div>
        </div>
        '''))
        display(widgets.HBox([w_urine],
                             layout=widgets.Layout(margin='0 0 8px 0')))
                             
        # ── Laboratory Values ──
        display(HTML(f'''
        <div style="
            background:linear-gradient(135deg,#f8f9fb 0%,#f2f4f7 100%);
            border-radius:12px;padding:10px 16px;
            margin-top:24px;margin-bottom:12px;
            border:1px solid #dde2ea;border-left:4px solid #8a9ab5;
            box-shadow:0 2px 8px rgba(0,0,0,0.05);
        ">
            <div style="font-size:0.85em;font-weight:700;color:{C_NAVY};
                        text-transform:uppercase;letter-spacing:0.06em;
                        padding-bottom:6px;border-bottom:1px solid #dde2ea;">
                Laboratory Values &nbsp;
                <span style="font-weight:400;font-size:0.9em;
                             color:{C_MUTED};text-transform:none">
                    leave blank or type NA if unavailable
                </span>
            </div>
        </div>
        '''))
        display(HTML(f'''
        <div style="
            background:white;
            border:1px solid #dde2ea;
            border-radius:10px;
            padding:14px 18px;
            margin-bottom:14px;
            font-size:0.82em;
        ">
            <div style="font-weight:700;color:{C_NAVY};margin-bottom:10px;
                        font-size:0.88em;text-transform:uppercase;letter-spacing:0.05em">
                Reference
            </div>
            <table style="width:100%;border-collapse:collapse;color:{C_MUTED}">
                <thead>
                    <tr style="background:#f4f6f9;font-size:0.8em;text-transform:uppercase;
                               letter-spacing:0.05em;color:{C_NAVY}">
                        <td style="padding:6px 8px;width:18%">Abbreviation</td>
                        <td style="padding:6px 8px;width:30%">Full Name</td>
                        <td style="padding:6px 8px;width:52%">Unit</td>
                    </tr>
                </thead>
                <tbody>
                    <tr style="border-bottom:1px solid #f0f2f5">
                        <td style="padding:5px 8px;font-weight:600;color:{C_TEXT}">Creatinine</td>
                        <td style="padding:5px 8px">Serum Creatinine</td>
                        <td style="padding:5px 8px">mg/dL</td>
                    </tr>
                    <tr style="border-bottom:1px solid #f0f2f5">
                        <td style="padding:5px 8px;font-weight:600;color:{C_TEXT}">Lactate</td>
                        <td style="padding:5px 8px">Serum Lactate</td>
                        <td style="padding:5px 8px">mmol/L</td>
                    </tr>
                    <tr style="border-bottom:1px solid #f0f2f5">
                        <td style="padding:5px 8px;font-weight:600;color:{C_TEXT}">Bilirubin</td>
                        <td style="padding:5px 8px">Total Bilirubin</td>
                        <td style="padding:5px 8px">mg/dL</td>
                    </tr>
                    <tr style="border-bottom:1px solid #f0f2f5">
                        <td style="padding:5px 8px;font-weight:600;color:{C_TEXT}">WBC</td>
                        <td style="padding:5px 8px">White Blood Cell Count</td>
                        <td style="padding:5px 8px">K/uL (10&sup3;/&micro;L)</td>
                    </tr>
                    <tr style="border-bottom:1px solid #f0f2f5">
                        <td style="padding:5px 8px;font-weight:600;color:{C_TEXT}">Hemoglobin</td>
                        <td style="padding:5px 8px">Hemoglobin</td>
                        <td style="padding:5px 8px">g/dL</td>
                    </tr>
                    <tr style="border-bottom:1px solid #f0f2f5">
                        <td style="padding:5px 8px;font-weight:600;color:{C_TEXT}">Platelets</td>
                        <td style="padding:5px 8px">Platelet Count</td>
                        <td style="padding:5px 8px">K/uL (10&sup3;/&micro;L)</td>
                    </tr>
                    <tr style="border-bottom:1px solid #f0f2f5">
                        <td style="padding:5px 8px;font-weight:600;color:{C_TEXT}">Sodium</td>
                        <td style="padding:5px 8px">Serum Sodium</td>
                        <td style="padding:5px 8px">mEq/L</td>
                    </tr>
                    <tr style="border-bottom:1px solid #f0f2f5">
                        <td style="padding:5px 8px;font-weight:600;color:{C_TEXT}">Potassium</td>
                        <td style="padding:5px 8px">Serum Potassium</td>
                        <td style="padding:5px 8px">mEq/L</td>
                    </tr>
                    <tr>
                        <td style="padding:5px 8px;font-weight:600;color:{C_TEXT}">BUN</td>
                        <td style="padding:5px 8px">Blood Urea Nitrogen</td>
                        <td style="padding:5px 8px">mg/dL</td>
                    </tr>
                </tbody>
            </table>
        </div>
        '''))
        display(widgets.VBox([
            widgets.HBox([w_cr,   w_cr_min,   w_cr_max]),
            widgets.HBox([w_lac,  w_lac_min,  w_lac_max]),
            widgets.HBox([w_bili, w_bili_min, w_bili_max]),
            widgets.HBox([w_wbc,  w_wbc_min,  w_wbc_max]),
            widgets.HBox([w_hgb,  w_hgb_min,  w_hgb_max]),
            widgets.HBox([w_plt,  w_plt_min,  w_plt_max]),
            widgets.HBox([w_na,   w_na_min,   w_na_max]),
            widgets.HBox([w_k,    w_k_min,    w_k_max]),
            widgets.HBox([w_bun,  w_bun_min,  w_bun_max]),
        ], layout=widgets.Layout(margin='0 0 24px 0')))

        display(assess_btn)
        display(HTML('<div style="margin:16px 0"></div>'))
        display(result_out)

In [12]:
hero_img = "Hospital.png"  # replace with your image

In [38]:
# ══════════════════════════════════════════════
# TAB ROUTING
# ══════════════════════════════════════════════
def switch_tab(page_name):
    current_page[0] = page_name
    if   page_name == 'home':       render_home()
    elif page_name == 'overview':   render_overview()
    elif page_name == 'assessment': render_assessment()
    elif page_name == 'data':       render_data()

def go_assessment(b=None):
    switch_tab('assessment')

tab_home.on_click(lambda b:     switch_tab('home'))
tab_overview.on_click(lambda b: switch_tab('overview'))
tab_data.on_click(lambda b:     switch_tab('data'))

# ── Start Assessment button ──
start_btn = widgets.Button(
    description='Start Patient Assessment',
    layout=widgets.Layout(width='260px', height='48px')
)
start_btn.style.button_color = C_ORANGE
start_btn.style.text_color   = 'white'
start_btn.style.font_weight  = '700'
start_btn.on_click(go_assessment)

# ── WELCOME / HOME SCREEN ──
def render_home():
    with page_output:
        clear_output()

        # ── HERO SECTION ──
        display(HTML(f'''
        <div style="
            background: linear-gradient(90deg, rgba(13,43,78,0.25) 0%, rgba(13,43,78,0.08) 40%, rgba(13,43,78,0.0) 70%);
            backdrop-filter: blur(4px);
            -webkit-backdrop-filter: blur(4px);
            padding: 56px 80px 60px 80px;
            display: flex;
            align-items: center;
            justify-content: space-between;
            gap: 48px;
            margin: -8px -8px 0 -8px;
            border-radius: 0 0 28px 28px;
        ">
            <!-- LEFT TEXT -->
            <div style="flex:1.2; max-width:560px">
                <div style="
                    font-family: 'Playfair Display', serif;
                    font-size: 2.8em;
                    font-weight: 600;
                    color: #0d2b4e;
                    line-height: 1.2;
                    margin-bottom: 20px;
                    text-shadow: 0 2px 12px rgba(0,0,0,0.2);
                ">
                    Comprehensive ICU Mortality Risk Analysis
                </div>
                <div style="
                    color: #4a5568;
                    font-size: 1.05em;
                    line-height: 1.75;
                    max-width: 460px;
                    margin-bottom: 28px;
                ">
                    Enter patient vitals and laboratory values to receive
                    an evidence-based mortality risk score with clinical
                    alerts and recommended actions.
                </div>
                <button onclick="
                    var btns = document.querySelectorAll('.widget-button');
                    for(var i=0;i<btns.length;i++){{
                        if(btns[i].textContent.trim()==='_assess_'){{
                            btns[i].click(); break;
                        }}
                    }}" style="
                    background:{C_ORANGE};
                    color:white;
                    border:none;
                    padding:14px 28px;
                    border-radius:8px;
                    font-weight:700;
                    font-size:1em;
                    cursor:pointer;
                    letter-spacing:0.02em;
                    box-shadow:0 4px 14px rgba(232,108,26,0.4);
                ">
                    Start Patient Assessment
                </button>
            </div>

            <!-- RIGHT IMAGE -->
            <div style="flex:0.9; text-align:center;">
                <img src="/voila/files/Hospital.png"
                    style="max-height:340px; max-width:100%;
                            object-fit:contain;
                            filter:drop-shadow(0 12px 32px rgba(0,0,0,0.2));">
            </div>
        </div>
        '''))

        # Hidden trigger button
        hidden_btn = widgets.Button(
            description='_assess_',
            layout=widgets.Layout(width='0px', height='0px', visibility='hidden')
        )
        hidden_btn.on_click(lambda b: switch_tab('assessment'))
        display(hidden_btn)

        # ── SECTION HEADING ──
        display(HTML(f'''
        <div style="text-align:center; margin:36px 0 24px 0">
            <div style="
                font-family:'Playfair Display',serif;
                font-size:1.55em;
                color:{C_NAVY};
                margin-bottom:8px;
                font-weight:600;
            ">What can you do here?</div>
            <div style="color:{C_MUTED};font-size:0.92em">
                Two tools to explore ICU mortality data and assess individual patients
            </div>
        </div>
        '''))

        # ── FEATURE CARDS ──
        display(HTML(f'''
        <div style="display:flex;gap:24px;margin-bottom:28px">

            <div style="
                flex:1;
                background:rgba(255,255,255,0.92);
                border-radius:16px;
                padding:32px 28px;
                border:1px solid {C_BORDER};
                box-shadow:0 4px 20px rgba(13,43,78,0.10);
                border-top:4px solid {C_BLUE};
            ">
                
                <div style="font-family:'Playfair Display',serif;font-size:1.2em;
                            color:{C_NAVY};margin-bottom:10px;font-weight:600">
                    Population Overview
                </div>
                <div style="color:{C_MUTED};font-size:0.87em;line-height:1.75;margin-bottom:18px">
                    Explore the MIMIC-IV ICU dataset across 47,291 patients.
                    View mortality rates by care unit and admission type, age
                    distributions, and median lab values split by outcome.
                </div>
                <div style="display:flex;flex-wrap:wrap;gap:7px">
                    <span style="background:{C_BLUE_LT};color:{C_BLUE};font-size:0.77em;
                                font-weight:600;padding:4px 12px;border-radius:20px">Outcome charts</span>
                    <span style="background:{C_BLUE_LT};color:{C_BLUE};font-size:0.77em;
                                font-weight:600;padding:4px 12px;border-radius:20px">Care unit breakdown</span>
                    <span style="background:{C_BLUE_LT};color:{C_BLUE};font-size:0.77em;
                                font-weight:600;padding:4px 12px;border-radius:20px">Lab medians</span>
                </div>
            </div>

            <div style="
                flex:1;
                background:rgba(255,255,255,0.92);
                border-radius:16px;
                padding:32px 28px;
                border:1px solid {C_BORDER};
                box-shadow:0 4px 20px rgba(13,43,78,0.10);
                border-top:4px solid #6a9fd8;
            ">
                
                <div style="font-family:'Playfair Display',serif;font-size:1.2em;
                            color:{C_NAVY};margin-bottom:10px;font-weight:600">
                    Data Explorer
                </div>
                <div style="color:{C_MUTED};font-size:0.87em;line-height:1.75;margin-bottom:18px">
                    Interactively explore any of 53 clinical features.
                    Compare distributions between survivors and non-survivors,
                    view correlation heatmaps, and identify missing data patterns.
                </div>
                <div style="display:flex;flex-wrap:wrap;gap:7px">
                    <span style="background:#eaf0fb;color:#1a5fa8;font-size:0.77em;
                                font-weight:600;padding:4px 12px;border-radius:20px">Distribution plots</span>
                    <span style="background:#eaf0fb;color:#1a5fa8;font-size:0.77em;
                                font-weight:600;padding:4px 12px;border-radius:20px">Correlation heatmap</span>
                    <span style="background:#eaf0fb;color:#1a5fa8;font-size:0.77em;
                                font-weight:600;padding:4px 12px;border-radius:20px">Missing data map</span>
                </div>
            </div>

            <div style="
                flex:1;
                background:rgba(255,255,255,0.92);
                border-radius:16px;
                padding:32px 28px;
                border:1px solid {C_BORDER};
                box-shadow:0 4px 20px rgba(13,43,78,0.10);
                border-top:4px solid {C_ORANGE};
            ">
                
                <div style="font-family:'Playfair Display',serif;font-size:1.2em;
                            color:{C_NAVY};margin-bottom:10px;font-weight:600">
                    Patient Assessment
                </div>
                <div style="color:{C_MUTED};font-size:0.87em;line-height:1.75;margin-bottom:18px">
                    Enter a patient's vitals and laboratory values — or mark
                    unavailable labs as NA. Get an instant 72h mortality risk score,
                    clinical alerts, and recommended interventions.
                </div>
                <div style="display:flex;flex-wrap:wrap;gap:7px">
                    <span style="background:{C_ORG_LT};color:{C_ORANGE};font-size:0.77em;
                                font-weight:600;padding:4px 12px;border-radius:20px">Risk gauge</span>
                    <span style="background:{C_ORG_LT};color:{C_ORANGE};font-size:0.77em;
                                font-weight:600;padding:4px 12px;border-radius:20px">Clinical alerts</span>
                    <span style="background:{C_ORG_LT};color:{C_ORANGE};font-size:0.77em;
                                font-weight:600;padding:4px 12px;border-radius:20px">Action items</span>
                </div>
            </div>

        </div>

        <div style="
            background:#f8f9fb;
            border:1px solid {C_BORDER};
            border-radius:10px;
            padding:14px 20px;
            font-size:0.82em;
            color:{C_MUTED};
            text-align:center;
            line-height:1.6;
            margin-bottom:32px;
        ">
            <b style="color:{C_TEXT}">Clinical Decision Support Only</b> &nbsp;&mdash;&nbsp;
            This tool is intended to assist, not replace, clinical judgment.
            All predictions should be interpreted in context by a qualified healthcare professional.
        </div>
        '''))
render_home()